In [ ]:
import numpy as np
import pandas as pd

# Document-level politeness annotations for the 480-email corpus.
# The original annotation dimensions:
# - Structural_Politeness_and_Politeness_Markers corresponds to Positive Face Saving.
# - Tone_and_Overall_Consideration corresponds to Negative Face Saving.

DATA_PATH = "data/annotation/480email_politeness_scores_of_the_annotators.csv"

df = pd.read_csv(DATA_PATH)

dims = {
    "Directness_vs_Indirectness": [
        "Directness_vs_Indirectness_admin",
        "Directness_vs_Indirectness_Brianna",
        "Directness_vs_Indirectness_Amber",
    ],
    "Positive_Face_Saving": [
        "Structural_Politeness_and_Politeness_Markers_admin",
        "Structural_Politeness_and_Politeness_Markers_Brianna",
        "Structural_Politeness_and_Politeness_Markers_Amber",
    ],
    "Negative_Face_Saving": [
        "Tone_and_Overall_Consideration_admin",
        "Tone_and_Overall_Consideration_Brianna",
        "Tone_and_Overall_Consideration_Amber",
    ],
}


def krippendorff_alpha(data, distance):
    """
    Compute Krippendorff's alpha.

    Parameters
    ----------
    data : array-like of shape (n_items, n_raters)
        Annotation matrix. Missing annotations should be represented by np.nan.

    distance : callable
        Distance function between two annotation values.

    Returns
    -------
    float
        Krippendorff's alpha.
    """
    data = np.asarray(data, dtype=float)

    # Observed disagreement
    observed_disagreement = 0.0
    n_pairs = 0
    pooled_values = []

    for row in data:
        values = row[~np.isnan(row)]
        pooled_values.extend(values.tolist())

        if len(values) >= 2:
            for i in range(len(values)):
                for j in range(i + 1, len(values)):
                    observed_disagreement += distance(values[i], values[j])
                    n_pairs += 1

    if n_pairs == 0:
        return np.nan

    Do = observed_disagreement / n_pairs

    # Expected disagreement from pooled annotation marginals
    unique_values, counts = np.unique(pooled_values, return_counts=True)
    probabilities = counts / counts.sum()

    De = sum(
        probabilities[i]
        * probabilities[j]
        * distance(unique_values[i], unique_values[j])
        for i in range(len(unique_values))
        for j in range(len(unique_values))
    )

    return 1.0 if De == 0 else 1.0 - (Do / De)


def interval_distance(a, b):
    """Squared distance for interval-scaled annotations."""
    return (a - b) ** 2


if __name__ == "__main__":
    for dimension_name, columns in dims.items():
        annotation_matrix = df[columns].to_numpy(dtype=float)

        usable_items = int(
            np.sum(
                np.sum(~np.isnan(annotation_matrix), axis=1) >= 2
            )
        )

        alpha = krippendorff_alpha(
            annotation_matrix,
            interval_distance,
        )

        print(
            f"{dimension_name}: "
            f"alpha_interval={alpha:.3f} | "
            f"usable_items={usable_items}"
        )